# 🤖 **Model Benchmark — BTC Target + ETH Cross-Asset + BTC Media**

### 🎯 **Objective**
Evaluate whether combining **ETH cross-asset features** and **BTC media features** improves **BTC crash prediction** beyond any single signal source.

### ⚠️ **Problem Definition**
Binary classification task on **BTC crashes**:

- `True` → BTC Crash (>10% drop within 7 days)
- `False` → No crash

### 🔬 **Benchmark Focus**
Test the **combined signal hypothesis**: ETH market dynamics + BTC media article volume together may provide better coverage of BTC crash risk than either alone. This is the richest BTC feature set tested.

### 📊 **Feature Set**

| Source | Features | Count |
|---|---|---|
| BTC market (notebook 01 EDA) | 17 standard market features | 17 |
| ETH cross-asset — engineered only (notebook 06 EDA) | 17 ETH features (raw OHLCV excluded) | 17 |
| BTC media (notebook 04 EDA) | `article_count_ma_3`, `article_count`, `article_count_lag_1`, `tone_ma_30`, `tone_ma_7`, `tone_lag_7`, `tone_lag_1` | 7 |
| **Total** | | **41** |

### 🚀 **Goal**
Determine whether combining ETH cross-asset + BTC media enrichment produces synergistic improvement over the individual signal sources. Raw OHLCV ETH features excluded (non-stationary); 17 engineered ETH features used.

In [ ]:
import pandas as pd
from pipelines.ml_benchmark import run_benchmark_pipeline

### **1. Load BTC, ETH Price and BTC Media Data**

In [ ]:
df_btc = pd.read_csv("../../data/gold/market/btc_usdt_1d_features.csv")
df_eth = pd.read_csv("../../data/gold/market/eth_usdt_1d_features.csv")
df_btc_media = pd.read_csv("../../data/gold/bitcoin_tone_gold.csv")

print(f"BTC price : {df_btc.shape[0]} rows, {df_btc.shape[1]} columns")
print(f"ETH price : {df_eth.shape[0]} rows, {df_eth.shape[1]} columns")
print(f"BTC media : {df_btc_media.shape[0]} rows, {df_btc_media.shape[1]} columns")

### **2. Merge BTC + ETH Price (inner join)**

In [ ]:
SELECTED_FEATURES = [
    # BTC market features (17)
    "return_1d", "return_7d",
    "volatility_7d", "volatility_30d",
    "buy_pressure",
    "drawdown",
    "ma_ratio",
    "lag_return_1d", "lag_return_7d",
    "lag_volatility_7d",
    "lag_buy_pressure",
    "lag_volume_norm",
    "momentum_acc",
    "momentum_volatility",
    "volume",
    "number_of_trades",
    "quote_asset_volume",

    # ETH cross-features — engineered only (17, mirrors BTC feature categories)
    # Raw OHLCV (high/open/low/close/volume) and absolute MAs (ma_7/ma_30) excluded
    "momentum_volatility_eth", "return_7d_eth", "lag_return_7d_eth",
    "volatility_30d_eth", "volatility_7d_eth", "lag_volatility_7d_eth",
    "lag_volume_norm_eth", "pressure_x_return_eth", "volume_norm_eth",
    "return_1d_eth", "lag_return_1d_eth", "drawdown_eth",
    "ma_ratio_eth", "volatility_ratio_eth",
    "buy_pressure_eth", "lag_buy_pressure_eth", "momentum_acc_eth",

    # BTC media enrichment (7) — from 04_eda_btc_price_and_media.ipynb
    "article_count_ma_3",
    "article_count",
    "article_count_lag_1",
    "tone_ma_30",
    "tone_ma_7",
    "tone_lag_7",
    "tone_lag_1",
]

TARGET = "target"

df = df[SELECTED_FEATURES + [TARGET]]

print(f"Dataset shape : {df.shape}")
print(f"Target distribution:\n{df[TARGET].value_counts()}")
df.isna().sum().loc[lambda s: s > 0]

### **3. Add BTC Media (inner join)**

In [ ]:
df_btc_media["date"] = pd.to_datetime(df_btc_media["date"], errors="coerce")
df_btc_media = (
    df_btc_media
    .dropna(subset=["date"])
    .sort_values("date")
    .drop_duplicates("date", keep="last")
    .reset_index(drop=True)
)

min_date = df["open_time"].min()
max_date = df["open_time"].max()
df_btc_media = df_btc_media[
    (df_btc_media["date"] >= min_date) & (df_btc_media["date"] <= max_date)
].copy()

df = pd.merge(df, df_btc_media, left_on="open_time", right_on="date", how="inner")
df.drop(columns=["open_time", "date"], inplace=True)

print(f"After adding BTC media : {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### **4. Feature Selection**

- **17 BTC market** — from `01_model_benchmark_btc.ipynb`
- **28 ETH cross-features** — from `06_eda_btc_with_eth.ipynb` (all 28 passed threshold, none redundant)
- **7 BTC media** — from `04_eda_btc_price_and_media.ipynb` (`avg_tone` excluded: score < 0.10)

In [ ]:
SELECTED_FEATURES = [
    # BTC market features (17)
    "return_1d", "return_7d",
    "volatility_7d", "volatility_30d",
    "buy_pressure",
    "drawdown",
    "ma_ratio",
    "lag_return_1d", "lag_return_7d",
    "lag_volatility_7d",
    "lag_buy_pressure",
    "lag_volume_norm",
    "momentum_acc",
    "momentum_volatility",
    "volume",
    "number_of_trades",
    "quote_asset_volume",

    # ETH cross-features — engineered only (17, mirrors BTC feature categories)
    # Raw OHLCV (high/open/low/close/volume) and absolute MAs (ma_7/ma_30) excluded
    "momentum_volatility_eth", "return_7d_eth", "lag_return_7d_eth",
    "volatility_30d_eth", "volatility_7d_eth", "lag_volatility_7d_eth",
    "lag_volume_norm_eth", "pressure_x_return_eth", "volume_norm_eth",
    "return_1d_eth", "lag_return_1d_eth", "drawdown_eth",
    "ma_ratio_eth", "volatility_ratio_eth",
    "buy_pressure_eth", "lag_buy_pressure_eth", "momentum_acc_eth",

    # BTC media enrichment (7) — from 04_eda_btc_price_and_media.ipynb
    "article_count_ma_3",
    "article_count",
    "article_count_lag_1",
    "tone_ma_30",
    "tone_ma_7",
    "tone_lag_7",
    "tone_lag_1",
]

TARGET = "target"

df = df[SELECTED_FEATURES + [TARGET]]

print(f"Dataset shape : {df.shape}")
print(f"Target distribution:\n{df[TARGET].value_counts()}")
df.isna().sum().loc[lambda s: s > 0]

### **5. Model Benchmark**

In [ ]:
results = run_benchmark_pipeline(df, verbose=False, plot_confusion=True)

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("pr_auc", ascending=False)

# Summary rows
results_df.loc["mean"] = results_df.select_dtypes("number").mean()
results_df.loc["std"]  = results_df.select_dtypes("number").std()

results_df